In [ ]:

!pip install tensorflow scikit-learn matplotlib numpy -q


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

np.random.seed(42)


In [ ]:

(x_train_full, y_train_full), (x_test_full, y_test_full) = mnist.load_data()

train_mask = (y_train_full == 0) | (y_train_full == 1)
test_mask = (y_test_full == 0) | (y_test_full == 1)

x_train, y_train = x_train_full[train_mask], y_train_full[train_mask]
x_test, y_test = x_test_full[test_mask], y_test_full[test_mask]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

print("Train samples:", x_train.shape[0])
print("Test samples:", x_test.shape[0])
print("Image shape:", x_train.shape[1:])


In [ ]:

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.suptitle("Sample Training Images (0 and 1)")
plt.show()


In [ ]:

clf = LogisticRegression(max_iter=1000)
clf.fit(x_train_flat, y_train)

y_pred = clf.predict(x_test_flat)
accuracy = accuracy_score(y_test, y_pred)
print(f"Discriminative Model (Logistic Regression) Accuracy: {accuracy * 100:.2f}%")


In [ ]:

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(x_test[i], cmap="gray")
    axes[0, i].set_title(f"True: {y_test[i]}")
    axes[0, i].axis("off")

    axes[1, i].imshow(x_test[i], cmap="gray")
    axes[1, i].set_title(f"Pred: {y_pred[i]}")
    axes[1, i].axis("off")
plt.suptitle("Discriminative Model: True Labels (top) vs Predictions (bottom)")
plt.tight_layout()
plt.show()


In [ ]:

input_dim = x_train_flat.shape[1]
encoding_dim = 32

input_img = Input(shape=(input_dim,))
encoded = Dense(128, activation="relu")(input_img)
encoded = Dense(encoding_dim, activation="relu")(encoded)

decoded = Dense(128, activation="relu")(encoded)
decoded = Dense(input_dim, activation="sigmoid")(decoded)

autoencoder = Model(input_img, decoded)
encoder = Model(input_img, encoded)

autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy")
autoencoder.summary()


In [ ]:

history = autoencoder.fit(
    x_train_flat, x_train_flat,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=1
)


In [ ]:

reconstructed = autoencoder.predict(x_test_flat[:8])

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(x_test_flat[i].reshape(28, 28), cmap="gray")
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")

    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap="gray")
    axes[1, i].set_title("Reconstructed")
    axes[1, i].axis("off")
plt.suptitle("Generative Model: Original vs Reconstructed Images")
plt.tight_layout()
plt.show()


In [ ]:

decoder_input = Input(shape=(encoding_dim,))
decoder_layer1 = autoencoder.layers[-2]
decoder_layer2 = autoencoder.layers[-1]
decoder_output = decoder_layer2(decoder_layer1(decoder_input))
decoder = Model(decoder_input, decoder_output)

encoded_imgs = encoder.predict(x_train_flat)
mean_vec = encoded_imgs.mean(axis=0)
std_vec = encoded_imgs.std(axis=0)

n_generate = 8
random_latent_vectors = np.random.normal(
    loc=mean_vec, scale=std_vec, size=(n_generate, encoding_dim)
)
generated_images = decoder.predict(random_latent_vectors)

fig, axes = plt.subplots(1, n_generate, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated_images[i].reshape(28, 28), cmap="gray")
    ax.set_title("Generated")
    ax.axis("off")
plt.suptitle("Newly Generated Digit Images (Sampled from Latent Space)")
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(3, 8, figsize=(14, 6))

for i in range(8):
    axes[0, i].imshow(x_test[i], cmap="gray")
    axes[0, i].set_title(f"Real ({y_test[i]})")
    axes[0, i].axis("off")

for i in range(8):
    axes[1, i].imshow(x_test[i], cmap="gray")
    axes[1, i].set_title(f"Pred: {y_pred[i]}")
    axes[1, i].axis("off")

for i in range(8):
    axes[2, i].imshow(generated_images[i].reshape(28, 28), cmap="gray")
    axes[2, i].set_title("Generated")
    axes[2, i].axis("off")

plt.suptitle("Row 1: Real Images | Row 2: Discriminative Predictions | Row 3: Generated Images")
plt.tight_layout()
plt.show()


In [ ]:

reflection = '''
Written Reflection:

1. The discriminative model (Logistic Regression) learned a decision boundary that
   separates the pixel patterns of digit 0 from digit 1, allowing it to classify
   new images with high accuracy ({:.2f}%).
2. The generative model (Autoencoder) learned the underlying structure and
   distribution of pixel intensities in the 0/1 digit images, compressing them
   into a latent representation and reconstructing/generating new plausible digit images.
3. The discriminative model answers "which class does this image belong to?" by
   modeling P(label | image), while the generative model answers "what does data
   in this class look like?" by modeling the underlying structure of P(image).
4. As a result, the discriminative model is optimized purely for classification
   accuracy, whereas the generative model is optimized to reconstruct and produce
   new data samples, even though it does not directly predict labels.
'''.format(accuracy * 100)

print(reflection)
